In [2]:
# Importowanie bibliotek
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from scipy.spatial.distance import mahalanobis
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Funkcja do obliczania miary Mahalanobisa
def calculate_mahalanobis(X, mean, cov_matrix):
    inv_cov_matrix = np.linalg.inv(cov_matrix)
    distances = [mahalanobis(x, mean, inv_cov_matrix) for x in X]
    return np.array(distances)

# Wczytanie danych
processed_filepath = './datasets/processed/'
data = pd.read_csv(processed_filepath + 'Thursday.csv')
df = data.copy()

# Wyświetlenie unikalnych wartości w kolumnie 'Label'
print("Unikalne wartości w kolumnie 'Label':")
print(df['Label'].unique())
print(df['Label'].value_counts())

# Przygotowanie danych
X = df.drop('Label', axis=1).values
y = df['Label'].values

# Normalizacja danych
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Podział na zbiory treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42, stratify=y)

# Obliczanie miary Mahalanobisa
mean = np.mean(X_train, axis=0)
cov_matrix = np.cov(X_train, rowvar=False)

# Regularizacja macierzy kowariancji
epsilon = 1e-10
cov_matrix += np.eye(cov_matrix.shape[0]) * epsilon

# Obliczanie odległości Mahalanobisa
mahalanobis_train = calculate_mahalanobis(X_train, mean, cov_matrix)
mahalanobis_test = calculate_mahalanobis(X_test, mean, cov_matrix)

# Dodanie miary Mahalanobisa jako nowej cechy
X_train = np.hstack((X_train, mahalanobis_train.reshape(-1, 1)))
X_test = np.hstack((X_test, mahalanobis_test.reshape(-1, 1)))

# Budowa modelu MLP
model = Sequential([
    Input((X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Kompilacja modelu
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Obliczanie wag klas (opcjonalne)
class_weights = {0: 0.1, 1: 300.0}

# Trening modelu
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), class_weight=class_weights)

# Predykcja i ewaluacja
y_pred_probs = model.predict(X_test)
y_pred_classes = (y_pred_probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_classes)
print("Macierz konfuzji:")
print(cm)
print(classification_report(y_test, y_pred_classes, target_names=['BENIGN', 'ATTACK']))

Unikalne wartości w kolumnie 'Label':
[0 1]
Label
0    456410
1      2216
Name: count, dtype: int64
Epoch 1/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 6s 468us/step - accuracy: 0.8657 - loss: 0.3312 - val_accuracy: 0.7878 - val_loss: 4.1487
Epoch 2/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 458us/step - accuracy: 0.8837 - loss: 1.0155 - val_accuracy: 0.8925 - val_loss: 4.6227
Epoch 3/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 472us/step - accuracy: 0.9051 - loss: 1.2076 - val_accuracy: 0.8539 - val_loss: 13.8056
Epoch 4/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 445us/step - accuracy: 0.8912 - loss: 1.8985 - val_accuracy: 0.9306 - val_loss: 5.6464
Epoch 5/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 436us/step - accuracy: 0.9080 - loss: 2.3977 - val_accuracy: 0.8752 - val_loss: 13.9970
Epoch 6/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 431us/step - accuracy: 0.9077 - loss: 3.2293 - val_accuracy: 0.9582 - val_loss: 8.5650
Epoch 7/10
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 5s 433us/step - accuracy: 0.8990 - loss: 4.9552 - 